# Spark DataFrame Practical Showcase

## Objective

This notebook demonstrates the most commonly used **PySpark DataFrame transformations** using a Quick Commerce (Blinkit-style) dataset.

By the end of this notebook, you will understand how to:

- Create a Spark Session
- Read CSV files from Amazon S3
- Define custom schemas
- Create and manipulate DataFrames
- Perform DataFrame transformations
- Apply Window Functions
- Execute Spark SQL queries
- Optimize joins using Broadcast Join

---

## Dataset

We will use three CSV files.

| File | Description |
|------|-------------|
| blinkit_orders.csv | Order level information |
| blinkit_order_items.csv | Products inside each order |
| blinkit_products.csv | Product master data |

These datasets are sufficient to demonstrate most Spark DataFrame transformations.

# 1. Spark Session and Path Setup

## Objective

Before working with DataFrames, we must initialize a **Spark Session**.

A Spark Session is the entry point for all Spark applications.

It provides APIs to:

- Create DataFrames
- Read data
- Execute Spark SQL
- Configure Spark
- Access SparkContext

In this section we will also define all S3 paths used throughout the notebook.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("QuickCommerceSparkShowcase").getOrCreate()

BUCKET_NAME = "spark-file-storage"

base_path = f"s3://{BUCKET_NAME}/spark-learning"
raw_path = f"{base_path}/blinkit/raw"
output_path = f"{base_path}/blinkit/output"

orders_csv_path = f"{raw_path}/blinkit_orders.csv"
order_items_csv_path = f"{raw_path}/blinkit_order_items.csv"
products_csv_path = f"{raw_path}/blinkit_products.csv"

print("Raw data path:", raw_path)
print("Output path:", output_path)
print("Spark version:", spark.version)

## Explanation

### SparkSession

Creates or retrieves an existing Spark application.

```python
SparkSession.builder.getOrCreate()
```

---

### appName()

Assigns a readable name to the Spark application.

---

### S3 Paths

Instead of hardcoding file locations multiple times, we define reusable variables.

Benefits

- Easy maintenance
- Cleaner code
- Reusable notebook

# 2. Read CSV Without Custom Schema

## Objective

The simplest way to load a CSV file is by allowing Spark to infer everything automatically.

Spark will

- Read column names
- Infer all columns as StringType by default unless instructed otherwise

This method is useful for quick exploration but is **not recommended for production workloads**.

In [ ]:
orders_without_schema_df = spark\
                            .read\
                            .option("header", "true")\
                            .csv(orders_csv_path)

orders_without_schema_df.printSchema()
orders_without_schema_df.show(5, truncate=False)

## Key Points

### Advantages

- Very easy to use
- Good for quick exploration

### Disadvantages

- Incorrect data types
- Extra transformations required later
- Not suitable for production pipelines

---

# 3. Read CSV Using Infer Schema

## Objective

Instead of treating every column as a String, Spark can inspect the dataset and infer the data type automatically.

This is achieved using

```python
.option("inferSchema", True)
```

Spark scans the data before creating the DataFrame.

Although this produces better data types, it requires an additional scan of the dataset.

In [ ]:
orders_inferred_df = spark\
                        .read\
                        .option("header", "true")\
                        .option("inferSchema", "true")\
                        .option("multiline", "true")\
                        .option("escape", '"')\
                        .option("mode", "PERMISSIVE")\
                        .csv(orders_csv_path)

orders_inferred_df.printSchema()
orders_inferred_df.show(5, truncate=False)

## CSV Options Explained

| Option | Description |
|----------|------------|
| header | Uses first row as column names |
| inferSchema | Automatically detects data types |
| multiLine | Allows records to span multiple lines |
| escape | Escapes quote characters |
| mode | Controls behaviour for malformed records |

### PERMISSIVE

Loads valid rows while assigning **NULL** to malformed values.

### FAILFAST

Stops the job immediately if an invalid record is encountered.

# 4. Read CSV Using Custom Schema

## Objective

For production-grade Spark applications, defining an explicit schema is the recommended approach.

A custom schema provides:

- Better performance
- Predictable data types
- No schema inference overhead
- Better documentation

In [ ]:
orders_schema = T.StructType([
    T.StructField("order_id", T.LongType(), False),
    T.StructField("customer_id", T.LongType(), True),
    T.StructField("order_date", T.StringType(), True),
    T.StructField("promised_delivery_time", T.StringType(), True),
    T.StructField("actual_delivery_time", T.StringType(), True),
    T.StructField("delivery_status", T.StringType(), True),
    T.StructField("order_total", T.DoubleType(), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("delivery_partner_id", T.LongType(), True),
    T.StructField("store_id", T.LongType(), True)
])

orders_df = spark\
                .read\
                .option("header", "true")\
                .schema(orders_schema)\
                .csv(orders_csv_path)

orders_df.printSchema()
orders_df.show(5, truncate=False)

## Why Use a Custom Schema?

### Advantages

- Eliminates schema inference overhead
- Ensures consistent data types
- Improves application performance
- Makes the code self-documenting

### Recommended Practice

Always use an explicit schema for production ETL pipelines.

# 5. Read the Supporting CSV Files

## Objective

In addition to the Orders dataset, we need two supporting datasets to perform
joins and business analysis.

These datasets help us answer questions like:

- Which products are generating the highest revenue?
- Which categories are selling the most?
- What is the total quantity sold for each product?

---

## Datasets

### 1. blinkit_order_items.csv

Contains product-level information for every order.

Each row represents one product purchased in an order.

Example

| order_id | product_id | quantity | unit_price |
|----------|------------|----------|-----------|

---

### 2. blinkit_products.csv

Contains product master information.

Example

| product_id | product_name | category | brand | price |

---

## Why Use a Custom Schema?

Instead of relying on Spark to infer data types, we explicitly define the schema to:

- Improve performance
- Ensure consistent data types
- Make the code self-documenting
- Avoid schema inference overhead

---

## Expected Output

After reading both files, we will have two DataFrames:

- `order_items_df`
- `products_df`

These DataFrames will be used later for joins, aggregations, window functions, and Spark SQL examples.

In [ ]:
order_items_schema = T.StructType([
    T.StructField("order_id", T.LongType(), False),
    T.StructField("product_id", T.LongType(), False),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DoubleType(), True),
])

products_schema = T.StructType([
    T.StructField("product_id", T.LongType(), False),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("brand", T.StringType(), True),
    T.StructField("price", T.DoubleType(), True),
    T.StructField("mrp", T.DoubleType(), True),
    T.StructField("margin_percentage", T.DoubleType(), True),
    T.StructField("shelf_life_days", T.IntegerType(), True),
    T.StructField("min_stock_level", T.IntegerType(), True),
    T.StructField("max_stock_level", T.IntegerType(), True)
])

order_items_df = (
    spark.read
         .option("header", True)
         .option("mode", "PERMISSIVE")
         .option("escape", '"')
         .schema(order_items_schema)
         .csv(order_items_csv_path)
)

products_df = (
    spark.read
         .option("header", True)
         .option("mode", "PERMISSIVE")
         .option("escape", '"')
         .schema(products_schema)
         .csv(products_csv_path)
)

order_items_df.show(5, truncate=False)
products_df.show(5, truncate=False)

## Key Takeaways

- Both datasets are loaded using explicit schemas.
- `order_items_df` contains transactional data.
- `products_df` contains master data.
- These DataFrames are commonly joined using `product_id`.

# 6. Select Transformation

## Objective

The `select()` transformation is used to retrieve specific columns from a DataFrame.

Instead of carrying every column throughout the pipeline, selecting only the required columns improves:

- Readability
- Performance
- Memory usage

---

## Syntax

```python
df.select(column1, column2, ...)
```

The `select()` method can also:

- Rename columns using `alias()`
- Perform expressions
- Apply Spark SQL functions
- Create derived columns

In [ ]:
orders_selected_df = orders_df\
                        .select(
                            F.col("order_id").alias("order_key"),
                            F.col("customer_id").alias("customer_key"),
                            F.col("store_id").alias("dark_store_id"),
                            F.col("delivery_status").alias("sla_status"),
                            F.col("order_total").alias("gross_order_amount"),
                            F.round(F.col("order_total"), 0).cast("long").alias("rounded_order_amount"),
                            F.concat_ws("-", F.col("store_id"), F.col("order_id")).alias("store_order_reference")
                        )
                        
orders_selected_df.printSchema()
orders_selected_df.show(10, truncate=False)

## Explanation

This example demonstrates multiple capabilities of `select()`:

- Selecting required columns
- Renaming columns with `alias()`
- Rounding numeric values
- Type casting
- Creating a new reference column using `concat_ws()`

Unlike `withColumn()`, the `select()` transformation returns only the specified columns.

# 7. withColumn()

## Objective

The `withColumn()` transformation is used to:

- Add a new column
- Replace an existing column

Unlike `select()`, it retains all existing columns while appending or replacing the specified column.

---

## Business Scenario

The source dataset stores timestamps as strings.

To support analytics, we need to:

- Convert strings into timestamps
- Extract the order date
- Extract the order month
- Extract the order hour
- Calculate delivery delays
- Estimate platform fees
- Estimate seller payouts

In [ ]:
amount_col = F.coalesce(F.col("order_total"), F.lit(0.0))

timestamp_pattern = 'yyyy-MM-dd HH:mm:ss'

orders_with_date_df = orders_df\
                            .withColumn("order_ts", F.to_timestamp(F.col("order_date"), timestamp_pattern))\
                            .withColumn("promised_delivery_ts", F.to_timestamp(F.col("promised_delivery_time"), timestamp_pattern))\
                            .withColumn("actual_delivery_ts", F.to_timestamp(F.col("actual_delivery_time"), timestamp_pattern))\
                            .withColumn("order_dt", F.to_date(F.col("order_ts")))\
                            .withColumn("order_month", F.date_format(F.col("order_dt"), "yyyy-MM"))\
                            .withColumn("order_hour", F.hour(F.col("order_ts")))\
                            .withColumn("delivery_delay_minutes", 
                                            F.round(
                                                (F.col("actual_delivery_ts").cast("long") - F.col("promised_delivery_ts").cast("long"))/60, 
                                                2
                                            )
                                        )\
                            .withColumn("platform_fee", F.round(amount_col * F.lit(0.02), 2))\
                            .withColumn("estimated_seller_payout", F.round(amount_col - F.col("platform_fee"), 2))

orders_with_date_df.printSchema()
orders_with_date_df.show(10, truncate=False)

## Functions Used

| Function | Purpose |
|----------|---------|
| `withColumn()` | Add or replace a column |
| `to_timestamp()` | Convert string to timestamp |
| `to_date()` | Extract date from timestamp |
| `date_format()` | Format a date or timestamp |
| `hour()` | Extract the hour from a timestamp |
| `coalesce()` | Return the first non-null value |
| `round()` | Round numeric values |
| `lit()` | Create a constant value |

---

## Key Takeaways

- `withColumn()` preserves all existing columns.
- Multiple `withColumn()` calls can be chained together.
- It is commonly used for feature engineering and ETL transformations.

# 8. withColumnRenamed()

## Objective

The `withColumnRenamed()` transformation changes the name of an existing column.

This is useful when preparing cleaner, business-friendly datasets for reporting or downstream processing.

---

## Syntax

```python
df.withColumnRenamed("old_name", "new_name")
```

Multiple column renames can be chained together.

In [ ]:
orders_renamed_df = orders_df\
                        .withColumnRenamed("order_id", "order_key")\
                        .withColumnRenamed("store_id", "dark_store_id")\
                        .withColumnRenamed("delivery_status", "sla_status")\
                        .withColumnRenamed("order_total", "gross_order_amount")
                        
orders_renamed_df.select(
    "order_key",
    "customer_id",
    "dark_store_id",
    "sla_status",
    "gross_order_amount"
).show(10, truncate=False)

## Key Takeaways

- `withColumnRenamed()` changes only the column name.
- It does not modify the column values or data type.
- Renaming columns improves readability and aligns datasets with business terminology.
- Chaining multiple `withColumnRenamed()` calls is a clean way to rename several columns in one transformation.

# 9. Filter Transformation

## Objective

The `filter()` transformation is used to retrieve only the rows that satisfy one or more conditions.

Filtering is one of the most frequently used transformations in Spark because it reduces the amount of data processed in downstream operations.

---

## Business Scenario

Suppose a business analyst wants to analyze only **successful on-time deliveries**.

Instead of processing the complete orders dataset, we can filter the DataFrame to include only:

- Orders delivered **On Time**
- Orders with a valid order amount
- Orders where the order amount is greater than zero
- Orders paid using supported payment methods

---

## Syntax

```python
df.filter(condition)

# OR

df.where(condition)
```

> **Note:** `filter()` and `where()` are aliases. Both perform the same operation.

In [ ]:
on_time_orders_df = orders_df\
                    .filter(
                        (F.col("delivery_status") == F.lit("On Time")) &
                        (F.col("order_total").isNotNull()) &
                        (F.col("order_total") > F.lit(0)) &
                        (F.col("payment_method").isin("UPI", "Card", "Wallet", "Cash"))
                    )
                    
on_time_orders_df.select(
    "order_id",
    "store_id",
    "delivery_status",
    "payment_method",
    "order_total"
).show(10, truncate=False)

## Explanation

This filter applies four conditions:

### 1. Delivery Status

```python
delivery_status == "On Time"
```

Keeps only on-time deliveries.

---

### 2. Check for NULL Values

```python
order_total.isNotNull()
```

Removes records where the order amount is missing.

---

### 3. Positive Order Amount

```python
order_total > 0
```

Excludes invalid or cancelled orders with zero or negative values.

---

### 4. Payment Method

```python
payment_method.isin(...)
```

Keeps only orders paid through the specified payment methods.

---

## Key Takeaways

- `filter()` returns only rows matching the condition.
- Multiple conditions are combined using `&` (AND) and `|` (OR).
- Always wrap each condition inside parentheses.

# 10. Drop Transformation

## Objective

The `drop()` transformation removes one or more columns from a DataFrame.

Removing unnecessary columns reduces the size of the DataFrame and simplifies downstream processing.

---

## Business Scenario

Suppose we are preparing a reporting dataset for store performance.

The `payment_method` column is not required for this report, so we remove it.

In [ ]:
orders_reporting_base_df = orders_df\
    .drop("payment_method")
    
orders_reporting_base_df.printSchema()
orders_reporting_base_df.show(5, truncate=False)

## Explanation

`drop()` removes the specified column(s) from the DataFrame.

Unlike SQL's `DROP COLUMN`, this operation does **not** modify the original DataFrame. It creates a new DataFrame without the specified columns.

---

## Syntax

```python
df.drop("column_name")
```

Remove multiple columns:

```python
df.drop("col1", "col2", "col3")
```

---

## Key Takeaways

- Removes one or more columns.
- Original DataFrame remains unchanged.
- Commonly used before exporting or reporting.

# 11. DropDuplicates Transformation

## Objective

The `dropDuplicates()` transformation removes duplicate records from a DataFrame.

Duplicate records can lead to:

- Incorrect revenue calculations
- Duplicate customer counts
- Incorrect reporting

---

## Business Scenario

Suppose the orders file accidentally contains duplicate `order_id` values.

To ensure accurate reporting, we remove duplicate orders before performing aggregations.

In [ ]:
orders_deduped_df = orders_df\
                    .dropDuplicates(["order_id"])
                    
duplicate_summary_rows = [
    ("Before Dropduplicates", orders_df.count()),
    ("After DropDuplicates", orders_deduped_df.count())
]

duplicate_summary_cols = ["Stage", "Row Count"]

duplicate_summary_df = spark.createDataFrame(duplicate_summary_rows, duplicate_summary_cols)

duplicate_summary_df.show(truncate=False)

## Explanation

### Remove duplicates based on specific columns

```python
df.dropDuplicates(["order_id"])
```

Only one record is retained for each unique `order_id`.

---

### Remove duplicates from the entire row

```python
df.dropDuplicates()
```

or

```python
df.distinct()
```

Both methods compare all columns in the DataFrame.

---

## Key Takeaways

- `dropDuplicates()` removes duplicate records.
- Use a business key such as `order_id` whenever possible.
- `distinct()` is equivalent to `dropDuplicates()` with no arguments.

# 12. orderBy Transformation

## Objective

The `orderBy()` transformation sorts rows in ascending or descending order.

Sorting is commonly used when:

- Displaying reports
- Finding top or bottom records
- Ranking data
- Preparing ordered outputs

---

## Business Scenario

Suppose the business wants to identify the **highest-value orders**.

We sort the dataset by:

1. Highest order amount (descending)
2. Oldest order date (ascending)

In [ ]:
orders_by_amount_df = orders_df\
                        .orderBy(
                            F.col("order_total").desc_nulls_last(),
                            F.col("order_date").asc_nulls_last()
                        )
                        
orders_by_amount_df.select(
    F.col("order_id"),
    F.col("store_id"),
    F.col("order_date"),
    F.col("delivery_status"),
    F.col("order_total").alias("gross_order_amount")
).show(10, truncate=False)

## Explanation

### Descending Sort

```python
desc()
```

Sorts values from highest to lowest.

---

### Ascending Sort

```python
asc()
```

Sorts values from lowest to highest.

---

### Handling NULL Values

Spark provides helper functions:

```python
asc_nulls_last()
desc_nulls_last()
asc_nulls_first()
desc_nulls_first()
```

These determine where NULL values appear in the sorted result.

---

## Key Takeaways

- `orderBy()` sorts the entire DataFrame.
- Multiple columns can be used for sorting.
- Explicitly specifying NULL ordering improves readability and predictability.

# 13. sort() Transformation

## Objective

The `sort()` transformation arranges the rows of a DataFrame in ascending or descending order.

In PySpark, `sort()` is an alias for `orderBy()`. Both methods produce the same result.

Sorting is commonly used to:

- Identify top-performing products
- Rank customers by spending
- Find highest or lowest order values
- Prepare reports for business users

---

## Syntax

```python
df.sort("column_name")

df.sort(F.col("column_name").desc())
```

You can sort by one or multiple columns.

---

## Business Scenario

Suppose the operations team wants to identify the highest-value orders first.

We sort the DataFrame by:

1. Order Total (Highest to Lowest)
2. Order Date (Oldest First)

In [ ]:
orders_sorted_df = orders_df\
                    .sort(
                        F.col("order_total").desc(),
                        F.col("order_date").asc()
                    )
orders_sorted_df.select(
    "order_id",
    "customer_id",
    "order_total",
    "order_date",
    "delivery_status"
).show(10, truncate=False)                  

## Explanation

### Descending Sort

```python
F.col("order_total").desc()
```

Displays the highest-value orders first.

---

### Ascending Sort

```python
F.col("order_date").asc()
```

Orders records from oldest to newest.

---

## Difference Between `sort()` and `orderBy()`

There is **no functional difference**.

```python
df.sort(...)
```

is equivalent to

```python
df.orderBy(...)
```

Choose the method that improves code readability.

---

## Key Takeaways

- `sort()` is an alias for `orderBy()`.
- Supports multiple sorting columns.
- Can sort in ascending or descending order.

# 14. groupBy() Transformation

## Objective

The `groupBy()` transformation groups rows based on one or more columns and performs aggregate calculations.

It is one of the most frequently used transformations in data engineering and analytics.

---

## Common Aggregations

- count()
- sum()
- avg()
- min()
- max()
- collect_list()

---

## Business Scenario

Suppose Blinkit wants to analyze:

- Total revenue by store
- Number of orders per store
- Average order value
- Highest order amount

Grouping enables these business metrics.

In [ ]:
store_sales_df = orders_df\
                    .groupBy(F.col("store_id"))\
                    .agg(
                        F.count("*").alias("total_orders"),
                        F.sum("order_total").alias("total_revenue"),
                        F.avg("order_total").alias("average_order_value"),
                        F.max("order_total").alias("highest-order"),
                        F.min("order_total").alias("lowest_order")
                    )\
                    .orderBy(F.col("total_revenue").desc())
                    
store_sales_df.printSchema()
store_sales_df.show(truncate=False)

## Explanation

### Step 1

```python
groupBy("store_id")
```

Groups all records belonging to the same store.

---

### Step 2

Aggregate functions calculate business metrics.

| Function | Description |
|----------|-------------|
| count() | Number of records |
| sum() | Total revenue |
| avg() | Average value |
| max() | Highest value |
| min() | Lowest value |

---

### Step 3

Sort the aggregated result.

```python
orderBy(desc("total_revenue"))
```

Displays the highest-performing stores first.

---

## Output Example

| store_id | total_orders | total_revenue |
|----------|--------------|---------------|
| 101 | 125 | 38450 |
| 205 | 119 | 36120 |

---

## Key Takeaways

- `groupBy()` groups records before aggregation.
- Always follow `groupBy()` with an aggregate function.
- Widely used in reporting and dashboard development.

# 15. Conditional Columns using `when()` and `otherwise()`

## Objective

The `when()` function is used to implement conditional logic in PySpark.

It is equivalent to the SQL **CASE WHEN** expression.

---

## SQL Equivalent

```sql
CASE
    WHEN condition THEN value
    ELSE value
END
```

---

## PySpark Equivalent

```python
F.when(condition, value).otherwise(value)
```

---

## Business Scenario

Suppose the business wants to categorize orders based on their value.

| Order Amount | Category |
|--------------|----------|
| >= 1000 | Premium |
| >= 500 | Standard |
| < 500 | Budget |

This classification helps marketing teams design targeted campaigns.

In [ ]:
orders_category_df = orders_df\
    .withColumn(
        "order_category",
        F.when(F.col("order_total") >= F.lit(1000), "Premium")\
        .when(F.col("order_total") >= F.lit(500), "Standard")\
        .otherwise("Budget")
    )
    
orders_category_df.select(
    "order_id",
    "customer_id",
    "order_total",
    "order_category"
).show(10, truncate=False)

## Explanation

### First Condition

```python
.when(order_total >= 1000, "Premium")
```

Assigns **Premium** to orders worth ₹1000 or more.

---

### Second Condition

```python
.when(order_total >= 500, "Standard")
```

Assigns **Standard** to orders between ₹500 and ₹999.

---

### Default Condition

```python
.otherwise("Budget")
```

Assigns **Budget** to all remaining orders.

---

## Why Use `when()`?

Typical use cases include:

- Customer Segmentation
- Order Classification
- Risk Categories
- SLA Status
- Revenue Bands
- Loyalty Programs

---

## Key Takeaways

- `when()` is the PySpark equivalent of SQL `CASE WHEN`.
- Multiple conditions can be chained together.
- `otherwise()` acts as the default condition.
- Frequently used for creating business-friendly categorical columns.

# 16. Window Functions

## Objective

Window Functions perform calculations across a group of related rows while preserving every row in the original DataFrame.

Unlike `groupBy()`, which returns one row per group, Window Functions allow us to:

- Rank records
- Calculate running totals
- Compute cumulative averages
- Compare rows within the same group
- Access previous or next rows

---

## Why Window Functions?

Suppose each Blinkit store has hundreds of orders.

Business users may ask questions like:

- Which is the highest-value order for each store?
- Which customer placed the first order?
- What is the cumulative revenue of each store?
- Rank all orders by value within each store.

These questions require calculations within groups while keeping every individual record.

Window Functions solve this problem.

---

## General Syntax

```python
from pyspark.sql.window import Window

window_spec = (
    Window
        .partitionBy(...)
        .orderBy(...)
)

df.withColumn(
    "new_column",
    window_function.over(window_spec)
)
```

## Window Specification

A Window Specification defines how Spark groups and orders the data before applying a window function.

It consists of two main components:

### partitionBy()

Divides the data into independent groups.

Example:

```python
Window.partitionBy("store_id")
```

Each store is processed separately.

---

### orderBy()

Defines the order of rows inside each partition.

Example

```python
Window.partitionBy("store_id") \
      .orderBy("order_total")
```

Rows are sorted within each store before calculations are performed.

In [ ]:
store_window_specification = Window\
                                .partitionBy("store_id")\
                                .orderBy(F.col("order_total").desc())

## row_number()

### Objective

Assign a unique sequential number to each row within a partition.

Even when two rows have the same value, each row receives a different number.

---

### Business Scenario

Rank all orders inside each store based on order value.

The highest-value order should receive Rank 1.

In [ ]:
orders_row_number_df = orders_df\
                        .withColumn(
                            "row_number",
                            F.row_number().over(store_window_specification)
                        )
                        
orders_row_number_df.select(
    "store_id",
    "order_id",
    "order_total",
    "row_number"
).show(20, truncate=False)

## Example

Store 101

| Order | Amount | Row Number |
|--------|--------|------------|
| O10 | 850 | 1 |
| O15 | 850 | 2 |
| O08 | 720 | 3 |

Notice that duplicate values still receive different numbers.

---

## Key Takeaways

- Always unique
- No duplicate ranks
- Best choice when selecting Top-N records

## rank()

### Objective

Assign ranks while allowing ties.

If two rows have the same value, they receive the same rank.

The next rank is skipped.

Example

```
1000
1000
900
```

Ranks become

```
1
1
3
```

In [ ]:
orders_rank_df = orders_df\
                    .withColumn(
                        "rank",
                        F.rank().over(store_window_specification)
                    )
                    
orders_rank_df.select(
    "store_id",
    "order_id",
    "order_total",
    "rank"
).show()

## Key Takeaways

- Equal values receive the same rank.
- The next rank is skipped.
- Useful for competition-style rankings.

## dense_rank()

### Objective

Assign ranks without leaving gaps.

If duplicate values exist, the next rank continues sequentially.

Example

```
1000
1000
900
```

Ranks become

```
1
1
2
```

In [ ]:
orders_rank_df = orders_df\
                    .withColumn(
                        "ds_rank",
                        F.dense_rank().over(store_window_specification)
                    )
                    
orders_rank_df.select(
    "store_id",
    "order_id",
    "order_total",
    "ds_rank"
).show()

## Comparison

| Amount | row_number | rank | dense_rank |
|---------|-----------|------|------------|
| 1000 | 1 | 1 | 1 |
| 1000 | 2 | 1 | 1 |
| 900 | 3 | 3 | 2 |

---

### Which One Should You Use?

| Function | Best Use Case |
|----------|---------------|
| row_number | Top 1 record |
| rank | Competition ranking |
| dense_rank | Business reporting |

## Running Total

Window Functions can calculate cumulative values without collapsing rows.

Business Example

Calculate cumulative sales for every store throughout the day.

In [ ]:
running_window = Window\
                    .partitionBy("store_id")\
                    .orderBy("order_ts")
                    
orders_running_total_df = orders_with_date_df\
                            .withColumn(
                                "running_sales",
                                F.sum("order_total").over(running_window)
                            )
                            
orders_running_total_df.select(
    "store_id",
    "order_id",
    "order_total",
    "running_sales"
).show(truncate=False)

## Moving Average

A moving average calculates the average over a rolling window of rows.

Business Example

Calculate the average order value using the last three orders for each store.

This helps smooth short-term fluctuations in sales.

In [ ]:
moving_average_3_days_window = Window\
                        .partitionBy("store_id")\
                        .orderBy("order_ts")\
                        .rowsBetween(-2, 0)
                        
orders_moving_avg_3_days_df = orders_with_date_df\
                            .withColumn(
                                "moving_avg_3_days",
                                F.avg("order_total").over(moving_average_3_days_window)
                            )
orders_moving_avg_3_days_df.select(
    "store_id",
    "order_id",
    "order_total",
    "moving_avg_3_days"
).show(truncate=False)

## rowsBetween()

`rowsBetween(start, end)` defines the range of rows included in the window calculation.

Example:

```python
rowsBetween(-2, 0)
```

This means:

- Current row (`0`)
- Previous row (`-1`)
- Two rows before (`-2`)

The average is calculated across these three rows.

---

## Common Values

| Expression | Meaning |
|------------|---------|
| `rowsBetween(-2, 0)` | Previous 2 rows + current row |
| `rowsBetween(Window.unboundedPreceding, 0)` | From the first row to the current row |
| `rowsBetween(-1, 1)` | Previous, current, and next row |

# 17. Temporary Views

## Objective

A Temporary View allows us to query a DataFrame using SQL syntax.

Instead of writing DataFrame transformations, we can register the DataFrame as a SQL table and execute SQL queries using `spark.sql()`.

---

## Why Use Temporary Views?

Spark supports two ways of processing data:

1. DataFrame API
2. Spark SQL

Both use the same Spark execution engine (Catalyst Optimizer), so you can choose the style that best fits your use case.

---

## Business Scenario

Suppose the analytics team is familiar with SQL but not PySpark.

Instead of rewriting business logic using DataFrame transformations, we can expose the DataFrame as a temporary SQL table.

In [ ]:
orders_df.createOrReplaceTempView("orders")
order_items_df.createOrReplaceTempView("order_items")
products_df.createOrReplaceTempView("products")

## Explanation

### createOrReplaceTempView()

Registers a DataFrame as a temporary SQL table.

```python
df.createOrReplaceTempView("table_name")
```

If the view already exists, Spark replaces it with the latest DataFrame.

---

## Characteristics

- Exists only during the current Spark session.
- Accessible using `spark.sql()`.
- Automatically removed when the Spark application stops.

---

## Temporary View vs Global Temporary View

| Temporary View | Global Temporary View |
|---------------|----------------------|
| Session scoped | Shared across Spark sessions |
| Automatically removed | Exists until the Spark application ends |
| Accessed directly | Accessed using `global_temp.view_name` |

# 18. Spark SQL

## Objective

Spark SQL allows us to query DataFrames using standard SQL syntax.

Once a DataFrame is registered as a Temporary View, it behaves like a SQL table.

---

## Why Spark SQL?

Many organizations already use SQL for analytics.

Spark SQL allows data engineers and analysts to:

- Reuse existing SQL knowledge
- Write complex analytical queries
- Join multiple DataFrames
- Perform aggregations
- Filter and sort data

without converting everything into DataFrame transformations.

---

## General Syntax

```python
result_df = spark.sql("""
SELECT ...
FROM table_name
WHERE ...
GROUP BY ...
ORDER BY ...
""")
```

In [ ]:
orders_summary_df = spark.sql(
    """
        SELECT
            order_id,
            customer_id,
            store_id,
            order_total,
            delivery_status
        FROM orders
        WHERE order_total > 500
        ORDER BY order_total DESC
    """
)

orders_summary_df.show(truncate=False)

## Query Breakdown

### SELECT

Chooses the columns to return.

```sql
SELECT order_id, customer_id, order_total
```

---

### FROM

Specifies the temporary view to query.

```sql
FROM orders
```

---

### WHERE

Filters rows based on a condition.

```sql
WHERE order_total > 500
```

---

### ORDER BY

Sorts the result.

```sql
ORDER BY order_total DESC
```

## DataFrame API vs Spark SQL

Both approaches generate optimized execution plans using Spark's Catalyst Optimizer.

### DataFrame API

```python
orders_df.filter(F.col("order_total") > 500)
```

### Spark SQL

```sql
SELECT *
FROM orders
WHERE order_total > 500
```

Both approaches produce equivalent results.

---

## When to Use DataFrame API

- Complex ETL pipelines
- Dynamic transformations
- Python-based logic
- Integration with PySpark functions

---

## When to Use Spark SQL

- SQL-heavy analytics
- Dashboard queries
- Business reporting
- Teams familiar with SQL

## Best Practices

- Register DataFrames with meaningful view names.
- Use SQL aliases to improve query readability.
- Filter data as early as possible to reduce processing.
- Keep SQL queries well formatted using multiline strings.
- Drop or recreate temporary views when the underlying DataFrame changes.

# 19. Business SQL Query

## Objective

In real-world data engineering projects, we rarely query a single table.

Instead, we combine multiple datasets to generate business insights.

In this example, we join:

- Orders
- Order Items
- Products

to calculate business metrics such as:

- Total Revenue
- Quantity Sold
- Number of Orders
- Revenue by Product Category

---

## Business Scenario

Suppose the Blinkit management team wants to answer:

> **"Which product categories generate the highest revenue?"**

To answer this, we need to:

1. Join Orders with Order Items.
2. Join Order Items with Products.
3. Aggregate sales metrics by category.

This is a common reporting requirement in retail, e-commerce, and quick-commerce companies.

In [ ]:
category_sales_df = spark.sql("""
SELECT
    p.category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.quantity) AS total_quantity_sold,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue,
    ROUND(AVG(oi.unit_price), 2) AS average_selling_price
FROM orders o
INNER JOIN order_items oi
    ON o.order_id = oi.order_id
INNER JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
""")

category_sales_df.show(truncate=False)

## Query Breakdown

### Step 1 – Join Orders and Order Items

```sql
FROM orders o
INNER JOIN order_items oi
    ON o.order_id = oi.order_id
```

This links each order to its purchased products.

---

### Step 2 – Join Product Master

```sql
INNER JOIN products p
    ON oi.product_id = p.product_id
```

Adds product attributes such as:

- Product Name
- Category
- Brand

---

### Step 3 – Aggregate Business Metrics

```sql
GROUP BY p.category
```

Calculates category-level metrics.

---

### Step 4 – Sort Results

```sql
ORDER BY total_revenue DESC
```

Displays the highest revenue-generating categories first.

## Business Metrics Explained

| Metric | Description |
|---------|-------------|
| `total_orders` | Number of unique orders for the category |
| `total_quantity_sold` | Total quantity of products sold |
| `total_revenue` | Revenue generated by the category |
| `average_selling_price` | Average selling price of products in the category |

---

## Sample Output

| Category | Orders | Revenue |
|----------|--------|---------|
| Fruits | 1,280 | ₹5,24,000 |
| Dairy | 1,145 | ₹4,89,500 |
| Snacks | 980 | ₹3,97,200 |

*(Illustrative output for understanding.)*

---

## Key Takeaways

- Business reporting often requires joining multiple datasets.
- Aggregations transform transactional data into meaningful business metrics.
- SQL remains one of the most common languages for analytical reporting.

# 20. Broadcast Join

## Objective

A Broadcast Join is a Spark optimization technique used when one DataFrame is significantly smaller than the other.

Instead of shuffling both DataFrames across the cluster, Spark broadcasts the smaller DataFrame to every executor.

This reduces network communication and improves join performance.

---

## Why Broadcast Join?

Consider the following datasets:

| Dataset | Approximate Size |
|----------|------------------|
| Orders | 500 Million rows |
| Products | 10 Thousand rows |

Without broadcasting:

- Spark shuffles both DataFrames across the cluster.

With broadcasting:

- The small Products DataFrame is copied to each executor.
- The large Orders DataFrame is processed locally.
- Network shuffle is greatly reduced.

This leads to faster execution.

In [ ]:
from pyspark.sql.functions import broadcast

orders_products_df = order_items_df\
                        .join(
                            broadcast(products_df), 
                            on="product_id", 
                            how="inner"
                        )

orders_products_df.show(10, truncate=False)

## Explanation

### `broadcast()`

```python
broadcast(products_df)
```

Instructs Spark to distribute the smaller DataFrame to all executors.

---

### Join Operation

```python
.join(
    broadcast(products_df),
    on="product_id",
    how="inner"
)
```

Spark avoids shuffling the smaller DataFrame, improving join efficiency.

---

## Join Types

Spark supports multiple join types:

| Join Type | Description |
|-----------|-------------|
| `inner` | Returns matching rows from both DataFrames |
| `left` | Returns all rows from the left DataFrame and matching rows from the right |
| `right` | Returns all rows from the right DataFrame and matching rows from the left |
| `full` | Returns all rows from both DataFrames |
| `left_semi` | Returns matching rows from the left DataFrame only |
| `left_anti` | Returns non-matching rows from the left DataFrame only |

## When Should You Use Broadcast Join?

Use a Broadcast Join when:

- One DataFrame is much smaller than the other.
- The smaller DataFrame can fit comfortably in executor memory.
- The small DataFrame acts as a lookup or dimension table.

Examples:

- Product Master
- Customer Master
- Country Codes
- Currency Exchange Rates
- Calendar Table

---

## Advantages

- Reduces data shuffling.
- Improves join performance.
- Lowers network I/O.
- Faster query execution.

---

## Limitations

- Do not broadcast large DataFrames.
- Broadcasting very large datasets can increase executor memory usage and may lead to memory errors.

---

## Key Takeaways

- Broadcast Join is a performance optimization.
- Best suited for joining a large fact table with a small dimension table.
- The `broadcast()` function gives Spark a hint to replicate the smaller DataFrame to all executors.

# Notebook Summary

Congratulations! 🎉

In this notebook, you explored the core PySpark DataFrame transformations and Spark SQL features used in modern data engineering.

## Topics Covered

| Topic | Concept |
|--------|---------|
| 1 | Spark Session |
| 2 | Create DataFrames |
| 3 | Custom Schema |
| 4 | Read CSV Files |
| 5 | Read Supporting Datasets |
| 6 | `select()` |
| 7 | `withColumn()` |
| 8 | `withColumnRenamed()` |
| 9 | `filter()` |
| 10 | `drop()` |
| 11 | `dropDuplicates()` |
| 12 | `orderBy()` |
| 13 | `sort()` |
| 14 | `groupBy()` |
| 15 | `when()` / `otherwise()` |
| 16 | Window Functions |
| 17 | Temporary Views |
| 18 | Spark SQL |
| 19 | Business SQL Query |
| 20 | Broadcast Join |

## Skills Gained

By completing this notebook, you can now:

- Build DataFrames from different sources.
- Apply common DataFrame transformations.
- Write analytical Spark SQL queries.
- Use Window Functions for advanced analytics.
- Join datasets efficiently.
- Optimize joins using Broadcast Join.

These concepts form the foundation of production-grade Spark ETL pipelines and are frequently used in Azure Databricks, AWS EMR, Apache Spark, and other distributed data processing platforms.

| Topic             | Transformation              | Type                        | Shuffle Required?       | Why Use It?                             |
| ----------------- | --------------------------- | --------------------------- | ----------------------- | --------------------------------------- |
| Select            | `select()`                  | Narrow                      | ❌ No                    | Select required columns                 |
| Alias             | `alias()`                   | Narrow                      | ❌ No                    | Rename columns                          |
| withColumn        | `withColumn()`              | Narrow                      | ❌ No                    | Add or modify columns                   |
| withColumnRenamed | `withColumnRenamed()`       | Narrow                      | ❌ No                    | Rename columns                          |
| Filter            | `filter()`                  | Narrow                      | ❌ No                    | Keep required rows                      |
| Drop              | `drop()`                    | Narrow                      | ❌ No                    | Remove unwanted columns                 |
| DropDuplicates    | `dropDuplicates()`          | **Wide**                    | ✅ Usually               | Remove duplicate records                |
| orderBy           | `orderBy()`                 | **Wide**                    | ✅ Yes                   | Globally sort data                      |
| sort              | `sort()`                    | **Wide**                    | ✅ Yes                   | Globally sort data                      |
| groupBy           | `groupBy()`                 | **Wide**                    | ✅ Yes                   | Aggregate data                          |
| when              | `when()`                    | Narrow                      | ❌ No                    | Apply conditional logic                 |
| Window Functions  | `over(Window)`              | **Usually Wide**            | ✅ Usually               | Rank, running totals, moving averages   |
| Temp View         | `createOrReplaceTempView()` | Action (Metadata Operation) | ❌ No                    | Register DataFrame for SQL queries      |
| Spark SQL         | `spark.sql()`               | Depends on Query            | Depends                 | Execute SQL on Spark data               |
| Broadcast Join    | `broadcast()`               | Optimized Join              | ❌ For broadcasted table | Speed up joins with small lookup tables |
